# Parking Space YOLO OBB Pipeline - Colab

Colab-ready pipeline for rotated parking-bay detection. It ignores XML boxes for OBB training and uses the RD New GeoJSON polygons as ground truth.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


## 1. Install Dependencies

`ultralytics` is for YOLO OBB. `rasterio` reads GeoTIFF geotransforms. `shapely` handles polygon intersection and dedupe.

In [ ]:
!pip install -q ultralytics rasterio shapely pillow


## 2. Paths

Upload the project folder to `MyDrive/ParkingSpaceDetection`. Keep the filenames matched: `000000000041.tif`, `000000000041.png`, `000000000041.txt`.

In [ ]:
from pathlib import Path
import sys

BASE = Path('/content/gdrive/MyDrive/ParkingSpaceDetection')
RUN = BASE / 'GeoreferenceTest/202606121032215360267'
SCRIPTS = BASE / 'scripts'

TIF_DIR = RUN / 'images'
PNG_DIR = RUN / 'png'
GEOJSON = BASE / 'GeoreferenceTest/combined.geojson' # name of input file
OBB_LABEL_DIR = RUN / 'obb_labels'
DATASET_DIR = RUN / 'dataset'
RUNS_DIR = RUN / 'runs/obb'
TRAIN_PROJECT = RUNS_DIR
TRAIN_NAME = 'train'
TRAIN_DIR = TRAIN_PROJECT / TRAIN_NAME
BEST_MODEL = TRAIN_DIR / 'weights/best.pt'
LAST_MODEL = TRAIN_DIR / 'weights/last.pt'

print('BASE:', BASE)
print('TIF_DIR:', TIF_DIR, TIF_DIR.exists())
print('GEOJSON:', GEOJSON, GEOJSON.exists())
print('SCRIPTS:', SCRIPTS, SCRIPTS.exists())
print('TRAIN_DIR:', TRAIN_DIR)
print('BEST_MODEL:', BEST_MODEL)

## 3. Check Expected Files

In [ ]:
print('GeoTIFFs:', len(list(TIF_DIR.glob('*.tif'))))
print('Scripts:', sorted(p.name for p in SCRIPTS.glob('*.py')))


## 4. TIFF -> PNG

YOLO trains on PNG images. The GeoTIFFs stay in `images/` for georeferencing.

In [ ]:
!python {SCRIPTS / '01_tif_to_png.py'} \
  --input {TIF_DIR} \
  --output {PNG_DIR}


## 5. GeoJSON -> YOLO OBB Labels

This creates rotated labels from the GeoJSON polygons. XML is not used because XML only stores axis-aligned boxes.

In [ ]:
!python {SCRIPTS / '02_geojson_to_yolo_obb.py'} \
  --geojson {GEOJSON} \
  --tiles {TIF_DIR} \
  --output {OBB_LABEL_DIR} \
  --min-visible 0.70


## 6. Build Train/Val Dataset

In [ ]:
!python {SCRIPTS / '03_make_train_val_split.py'} \
  --images {PNG_DIR} \
  --labels {OBB_LABEL_DIR} \
  --dataset {DATASET_DIR} \
  --train-ratio 0.80 \
  --overwrite


## 7. Inspect Dataset

In [ ]:
# Self-contained dataset check/build cell.
# If dataset/data.yaml does not exist yet, this cell runs the split step first.

print('PNG_DIR:', PNG_DIR)
print('OBB_LABEL_DIR:', OBB_LABEL_DIR)
print('DATASET_DIR:', DATASET_DIR)

pngs = sorted(PNG_DIR.glob('*.png'))
labels = sorted(OBB_LABEL_DIR.glob('*.txt'))
print('source pngs:', len(pngs))
print('source obb labels:', len(labels))

if not pngs:
    raise RuntimeError('No PNGs found. Run the TIFF -> PNG cell first.')
if not labels:
    raise RuntimeError('No OBB labels found. Run the GeoJSON -> YOLO OBB Labels cell first.')

if not (DATASET_DIR / 'data.yaml').exists():
    print('data.yaml missing, building train/val dataset now...')
    !python {SCRIPTS / '03_make_train_val_split.py'} \
      --images {PNG_DIR} \
      --labels {OBB_LABEL_DIR} \
      --dataset {DATASET_DIR} \
      --train-ratio 0.80 \
      --overwrite

for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    folder_path = DATASET_DIR / folder
    print(folder, len(list(folder_path.glob('*'))))

print('\ndata.yaml:')
print((DATASET_DIR / 'data.yaml').read_text())


## 8. Train YOLO OBB

Default here is a fast CPU sanity run: `yolo11n-obb.pt`, 5 epochs, image size 640, batch 2. This is for testing that the pipeline works. For real training, switch to GPU and use `yolo11s-obb.pt`, more epochs, and `imgsz=1024`.


In [ ]:
# Fast sanity run. This should work on CPU, but it is only for testing the pipeline.
#TRAIN_MODEL = 'yolo11n-obb.pt'
#TRAIN_EPOCHS = 5
#TRAIN_IMGSZ = 640
#TRAIN_BATCH = 2
#TRAIN_DEVICE = 'cpu'

# For real training later, use something like:
TRAIN_MODEL = 'yolo11s-obb.pt'
TRAIN_EPOCHS = 100
TRAIN_IMGSZ = 1024
TRAIN_BATCH = 16
TRAIN_DEVICE = 0  # requires Colab GPU

print('training model:', TRAIN_MODEL)
print('training device:', TRAIN_DEVICE)
print('training output:', TRAIN_DIR)

!python {SCRIPTS / '04_train_obb.py'} \
  --data {DATASET_DIR / 'data.yaml'} \
  --model {TRAIN_MODEL} \
  --epochs {TRAIN_EPOCHS} \
  --imgsz {TRAIN_IMGSZ} \
  --batch {TRAIN_BATCH} \
  --device {TRAIN_DEVICE} \
  --project {TRAIN_PROJECT} \
  --name {TRAIN_NAME} \
  --exist-ok

## 9. Predict

Uses the trained `best.pt`. For the CPU sanity run, prediction also runs on CPU.


In [ ]:
PREDICT_SOURCE = DATASET_DIR / 'images/val'
PREDICT_PROJECT = RUNS_DIR
PREDICT_DEVICE = TRAIN_DEVICE if 'TRAIN_DEVICE' in globals() else 'cpu'

print('prediction device:', PREDICT_DEVICE)
print('Best model:', BEST_MODEL, BEST_MODEL.exists())

if not BEST_MODEL.exists():
    raise RuntimeError('No trained model found in Google Drive. The training cell must finish successfully first.')

!python {SCRIPTS / '05_predict_obb.py'} \
  --model {BEST_MODEL} \
  --source {PREDICT_SOURCE} \
  --project {PREDICT_PROJECT} \
  --name predict \
  --imgsz {TRAIN_IMGSZ if 'TRAIN_IMGSZ' in globals() else 640} \
  --conf 0.25 \
  --device {PREDICT_DEVICE}

## 9b. Show Prediction Images

Ultralytics saves rendered prediction images in the predict folder. This cell displays them in Colab.


In [ ]:
from IPython.display import Image, display

predict_dirs = sorted((RUN / 'runs/obb').glob('predict*'), key=lambda p: p.stat().st_mtime)
PREDICT_DIR = predict_dirs[-1]

print('using:', PREDICT_DIR)

image_files = sorted(
    list(PREDICT_DIR.glob('*.jpg')) +
    list(PREDICT_DIR.glob('*.jpeg')) +
    list(PREDICT_DIR.glob('*.png'))
)

print('prediction images:', len(image_files))
for image_path in image_files[:10]:
    print(image_path.name)
    display(Image(filename=str(image_path)))

In [ ]:
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display

COMPARE_DIR = RUN / 'gt_vs_pred_preview'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

predict_dirs = sorted(PREDICT_PROJECT.glob('predict*'), key=lambda p: p.stat().st_mtime)
if not predict_dirs:
    raise RuntimeError(f'No predict folders found under {PREDICT_PROJECT}')

PREDICT_DIR = predict_dirs[-1]
PRED_TXT_DIR = PREDICT_DIR / 'labels'

print('using prediction folder:', PREDICT_DIR)
print('prediction labels dir:', PRED_TXT_DIR, PRED_TXT_DIR.exists())

try:
    font = ImageFont.truetype('DejaVuSans.ttf', 24)
except Exception:
    font = None

def draw_obb_file(draw, label_path, image_size, color, width=4, show_conf=False):
    if not label_path.exists():
        return 0

    w, h = image_size
    count = 0

    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue

        parts = line.split()
        coords = [float(v) for v in parts[1:9]]
        points = [(coords[i] * w, coords[i + 1] * h) for i in range(0, 8, 2)]

        draw.line(points + [points[0]], fill=color, width=width)

        if show_conf and len(parts) >= 10:
            conf = float(parts[9])
            x, y = points[0]
            draw.text((x, y), f'{conf:.2f}', fill=color, font=font)

        count += 1

    return count

val_images = sorted((DATASET_DIR / 'images/val').glob('*.png'))
print('val images:', len(val_images))

for image_path in val_images[:10]:
    img = PILImage.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)

    gt_path = DATASET_DIR / 'labels/val' / f'{image_path.stem}.txt'
    pred_path = PRED_TXT_DIR / f'{image_path.stem}.txt'

    gt_count = draw_obb_file(draw, gt_path, img.size, color='red', width=4, show_conf=False)
    pred_count = draw_obb_file(draw, pred_path, img.size, color='lime', width=4, show_conf=True)

    draw.rectangle((10, 10, 430, 82), fill='black')
    draw.text((20, 18), f'red GT: {gt_count}', fill='red', font=font)
    draw.text((20, 48), f'green pred: {pred_count}', fill='lime', font=font)

    out_path = COMPARE_DIR / image_path.name
    img.save(out_path)

    print(out_path.name, 'GT:', gt_count, 'pred:', pred_count)
    display(img)

## 10. Predictions -> RD New GeoJSON

Prediction `.txt` files are converted back to RD New polygons using the same-name GeoTIFF.

In [ ]:
predict_dirs = sorted(PREDICT_PROJECT.glob('predict*'), key=lambda p: p.stat().st_mtime)
if not predict_dirs:
    raise RuntimeError(f'No predict folders found under {PREDICT_PROJECT}')

PREDICT_DIR = predict_dirs[-1]
PRED_TXT_DIR = PREDICT_DIR / 'labels'
PRED_GEOJSON = RUN / 'predictions.geojson'

print('using prediction labels:', PRED_TXT_DIR)

!python {SCRIPTS / '06_predictions_to_geojson.py'} \
  {PRED_TXT_DIR} \
  --tiles {TIF_DIR} \
  -o {PRED_GEOJSON}

## 11. Dedupe Overlapping Tile Predictions

Overlapping tiles can detect the same parking bay twice. Dedupe in RD New space with polygon IoU, keeping highest confidence.

In [ ]:
DEDUPED_GEOJSON = RUN / 'predictions_deduped.geojson'

!python {SCRIPTS / '07_dedupe_predictions.py'} \
  {PRED_GEOJSON} \
  -o {DEDUPED_GEOJSON} \
  --iou 0.50

print('Deduped output:', DEDUPED_GEOJSON)


In [ ]:
from ultralytics import YOLO

VAL_PROJECT = RUNS_DIR
VAL_NAME = 'val'

print('validating model:', BEST_MODEL, BEST_MODEL.exists())
model = YOLO(str(BEST_MODEL))
metrics = model.val(
    data=str(DATASET_DIR / 'data.yaml'),
    project=str(VAL_PROJECT),
    name=VAL_NAME,
    exist_ok=True,
    device=TRAIN_DEVICE if 'TRAIN_DEVICE' in globals() else None,
)
print(metrics)
print('validation output:', VAL_PROJECT / VAL_NAME)

## Notes

- CRS is `EPSG:28992 - Amersfoort / RD New`.
- Upload required data to `MyDrive/ParkingSpaceDetection/GeoreferenceTest/...`.
- Training outputs are saved in Google Drive at `GeoreferenceTest/202606121032215360267/runs/obb/train/`.
- The trained parking-space checkpoint is `runs/obb/train/weights/best.pt`.
- Prediction outputs are saved in Google Drive at `runs/obb/predict*/`.
- Validation outputs are saved in Google Drive at `runs/obb/val/`.
- The XML labels are optional for this notebook and are not used for OBB.
- The important link is the basename: `000000000041.png`, `000000000041.tif`, `000000000041.txt`.